# Thử TranslateGemma dịch sách Anh → Việt

Notebook này đọc các trang bạn chọn từ nhiều file PDF, tách thành từng đoạn văn, dịch bằng các cỡ
TranslateGemma (4B, 12B, 27B, nén 4-bit) rồi xuất ra **một file Excel** có cột câu gốc, cột bản dịch
của từng mô hình và các cột trống để bạn dán bản dịch của ChatGPT / Gemini vào so sánh.

## Chuẩn bị trước khi chạy (làm một lần)

1. **Xin quyền dùng mô hình:** đăng nhập Hugging Face, mở trang `google/translategemma-4b-it`
   (và bản 12b, 27b) rồi bấm chấp nhận điều khoản Gemma.
2. **Tạo token Hugging Face** (loại Read) trong phần Settings → Access Tokens.
3. Trong notebook Kaggle: **Add-ons → Secrets**, thêm secret tên `HF_TOKEN`, dán token vào và bật cho notebook này.
4. **Settings → Accelerator: GPU T4 x2**, và bật **Internet**.
5. **Add Input → Upload** các file PDF của bạn thành một dataset rồi gắn vào notebook.

Chạy lần đầu với `QUICK_TEST = True` (chỉ dịch 30 đoạn, bản 4B) để kiểm tra mọi thứ hoạt động,
sau đó đổi thành `False` để chạy cả bộ.

**Chạy cả bộ bằng Save Version, không bấm Run All:** chọn Save Version → Save & Run All (Commit).
Kaggle chạy ngầm trên máy chủ, tắt trình duyệt hay mất mạng cũng không sao, và file kết quả được giữ
lại trong tab Output của phiên bản đó.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes pymupdf openpyxl

## 1. Cấu hình

Chạy ô bên dưới một lần để xem notebook tìm thấy những file PDF nào và mỗi file có bao nhiêu trang,
rồi sửa `BOOKS` cho đúng tên file và khoảng trang bạn muốn thử. Số trang tính từ 1 theo thứ tự trong
file PDF (không phải số in trên trang sách).

In [ ]:
import glob, os

# Liet ke cac file PDF da gan vao notebook
import fitz  # PyMuPDF
found = sorted(glob.glob("/kaggle/input/**/*.pdf", recursive=True))
for p in found:
    with fitz.open(p) as d:
        print(f"{d.page_count:5d} trang  {p}")
if not found:
    print("Chua thay file PDF nao trong /kaggle/input - hay Add Input truoc.")

In [ ]:
# ---- SUA O DAY ---------------------------------------------------------
# Moi dong: ten file (chi can mot phan ten, khong phan biet hoa thuong) va cac khoang trang.
BOOKS = [
    {"file": "ten-sach-1", "pages": "30-37, 210-217, 400-407"},
    {"file": "ten-sach-2", "pages": "15-22, 120-127"},
]

QUICK_TEST = True          # True: chi dich 30 doan bang ban 4B de kiem tra ky thuat
MODELS = [                 # bo bot dong neu muon chay it mo hinh hon
    "google/translategemma-4b-it",
    "google/translategemma-12b-it",
    # "google/translategemma-27b-it",   # nang, de chay rieng sau khi 4B va 12B da xong
]
BATCH_SIZE = 4             # giam xuong 2 neu van bao het bo nho GPU
OUT_DIR = "/kaggle/working"
# -------------------------------------------------------------------------

if QUICK_TEST:
    MODELS = MODELS[:1]

## 2. Đọc PDF và tách đoạn

Mỗi khối văn bản trên trang được coi là một đoạn. Notebook bỏ qua các khối quá ngắn (tiêu đề, số trang)
và các khối phần lớn là ký hiệu (công thức, bảng số, code), vì chữ trích từ những khối đó thường vỡ và
không phản ánh chất lượng dịch. Số khối bị bỏ qua được in ra để bạn biết.

In [ ]:
import re
import pandas as pd

MIN_CHARS = 60      # doan ngan hon thi bo qua
MAX_CHARS = 1500    # doan dai hon thi cat theo cau
MIN_LETTER_RATIO = 0.70


def parse_pages(spec: str):
    """'30-37, 210, 400-402' -> [30, 31, ..., 37, 210, 400, 401, 402]"""
    pages = []
    for part in spec.split(","):
        part = part.strip()
        if not part:
            continue
        if "-" in part:
            a, b = (int(x) for x in part.split("-", 1))
            pages.extend(range(a, b + 1))
        else:
            pages.append(int(part))
    return pages


def clean_block(text: str) -> str:
    text = re.sub(r"-\n(?=[a-z])", "", text)   # noi tu bi ngat cuoi dong: transla-\ntion
    text = re.sub(r"\s*\n\s*", " ", text)      # xuong dong trong doan -> dau cach
    return re.sub(r"\s{2,}", " ", text).strip()


CONTROL_CHARS = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")


def looks_like_prose(text: str) -> bool:
    if len(text) < MIN_CHARS:
        return False
    if CONTROL_CHARS.search(text):   # chu trich tu font toan bi vo, khong dung de cham
        return False
    letters = sum(ch.isalpha() or ch.isspace() for ch in text)
    return letters / len(text) >= MIN_LETTER_RATIO


def split_long(text: str):
    if len(text) <= MAX_CHARS:
        return [text]
    parts, cur = [], ""
    for sent in re.split(r"(?<=[.!?])\s+", text):
        if cur and len(cur) + len(sent) + 1 > MAX_CHARS:
            parts.append(cur)
            cur = sent
        else:
            cur = f"{cur} {sent}".strip()
    if cur:
        parts.append(cur)
    return parts


def find_pdf(name_part: str, candidates):
    hits = [p for p in candidates if name_part.lower() in os.path.basename(p).lower()]
    if len(hits) != 1:
        raise ValueError(f"'{name_part}' khop {len(hits)} file PDF, can khop dung 1: {hits}")
    return hits[0]


def extract_segments(books, candidates):
    rows, skipped = [], 0
    for book in books:
        path = find_pdf(book["file"], candidates)
        title = os.path.splitext(os.path.basename(path))[0]
        with fitz.open(path) as doc:
            for page_no in parse_pages(book["pages"]):
                if not 1 <= page_no <= doc.page_count:
                    print(f"Bo qua trang {page_no} cua {title}: ngoai pham vi")
                    continue
                blocks = doc[page_no - 1].get_text("blocks")
                for b in sorted(blocks, key=lambda b: (round(b[1]), round(b[0]))):
                    if b[6] != 0:            # 0 = khoi van ban, 1 = hinh anh
                        continue
                    text = clean_block(b[4])
                    if not looks_like_prose(text):
                        skipped += 1
                        continue
                    for piece in split_long(text):
                        rows.append({"book": title, "page": page_no, "source": piece})
    df = pd.DataFrame(rows)
    df.insert(0, "id", range(1, len(df) + 1))
    print(f"Lay duoc {len(df)} doan, bo qua {skipped} khoi (qua ngan hoac chu yeu la ky hieu).")
    return df

In [ ]:
segments = extract_segments(BOOKS, found)
if QUICK_TEST:
    segments = segments.head(30).copy()
print(segments.groupby("book").agg(doan=("id", "count"), trang=("page", "nunique")))
segments.head(5)

## 3. Dịch bằng từng mô hình

Các mô hình được nén 4-bit để vừa GPU miễn phí. Kết quả được lưu sau mỗi mô hình, nên nếu phiên bị
ngắt bạn chạy lại ô này là nó bỏ qua những mô hình đã xong. Notebook cũng đo tốc độ (token mỗi giây)
của từng mô hình, là con số bạn cần để ước lượng thời gian dịch một chương.

In [ ]:
import gc, time, torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
CACHE_CSV = os.path.join(OUT_DIR, "ket_qua_tam.csv")


def load_model(model_id):
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                               bnb_4bit_compute_dtype=torch.bfloat16)
    processor = AutoProcessor.from_pretrained(model_id, token=HF_TOKEN)
    processor.tokenizer.padding_side = "left"
    model = AutoModelForImageTextToText.from_pretrained(
        model_id, device_map="auto", quantization_config=quant, token=HF_TOKEN)
    model.eval()
    return processor, model


def build_prompt(processor, text):
    messages = [{"role": "user", "content": [{
        "type": "text", "source_lang_code": "en", "target_lang_code": "vi", "text": text}]}]
    return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


@torch.inference_mode()
def translate_all(processor, model, texts, batch_size):
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]), reverse=True)
    out, new_tokens, started = [None] * len(texts), 0, time.time()
    for start in range(0, len(order), batch_size):
        idx = order[start:start + batch_size]
        prompts = [build_prompt(processor, texts[i]) for i in idx]
        enc = processor.tokenizer(prompts, return_tensors="pt", padding=True,
                                  add_special_tokens=False).to(model.device)
        limit = min(1024, int(enc["input_ids"].shape[1] * 2) + 64)
        gen = model.generate(**enc, max_new_tokens=limit, do_sample=False)
        gen = gen[:, enc["input_ids"].shape[1]:]
        pad_id = processor.tokenizer.pad_token_id
        new_tokens += int((gen != pad_id).sum())
        for i, ids in zip(idx, gen):
            out[i] = processor.tokenizer.decode(ids, skip_special_tokens=True).strip()
        done = min(start + batch_size, len(order))
        if done == len(order) or (start // batch_size) % 10 == 0:
            print(f"  {done}/{len(order)} doan, {(time.time() - started) / 60:.1f} phut", flush=True)
    elapsed = time.time() - started
    print(f"  | {elapsed / 60:.1f} phut, {new_tokens / elapsed:.1f} token/giay")
    return out, new_tokens / elapsed


def export_xlsx(df):
    from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE
    from openpyxl.styles import Alignment
    from openpyxl.utils import get_column_letter
    df = df.copy()
    for col in ["chatgpt", "gemini", "loai_loi", "ghi_chu"]:
        if col not in df.columns:
            df[col] = ""
    df = df.map(lambda v: ILLEGAL_CHARACTERS_RE.sub("", v) if isinstance(v, str) else v)
    path = os.path.join(OUT_DIR, "so_sanh_ban_dich.xlsx")
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name="so_sanh")
        ws = writer.sheets["so_sanh"]
        for i, name in enumerate(df.columns, start=1):
            wide = name not in ("id", "book", "page", "loai_loi")
            ws.column_dimensions[get_column_letter(i)].width = 55 if wide else 14
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = Alignment(wrap_text=True, vertical="top")
        ws.freeze_panes = "E2"
    print("Da luu:", path)
    return df


results = segments.copy()
if os.path.exists(CACHE_CSV):
    cached = pd.read_csv(CACHE_CSV)
    if list(cached["source"]) == list(results["source"]):
        results = cached
        print("Da nap ket qua tam tu lan chay truoc.")

speeds = {}
for model_id in MODELS:
    col = model_id.split("/")[-1].replace("translategemma-", "tg_").replace("-it", "")
    if col in results.columns and results[col].notna().all():
        print(f"{col}: da co, bo qua.")
        continue
    print(f"Dang nap {model_id} ...", flush=True)
    try:
        t0 = time.time()
        processor, model = load_model(model_id)
        print(f"  Nap xong sau {(time.time() - t0) / 60:.1f} phut, bat dau dich.", flush=True)
        results[col], speeds[col] = translate_all(processor, model, list(results["source"]), BATCH_SIZE)
        results.to_csv(CACHE_CSV, index=False)
        export_xlsx(results)   # xuat Excel ngay sau moi mo hinh, mat ket noi van con ket qua
    except Exception as err:   # vi du: het bo nho GPU voi ban 27B
        print(f"\n  LOI voi {model_id}: {type(err).__name__}: {err}")
    finally:
        processor = model = None
        gc.collect()
        torch.cuda.empty_cache()

print("Toc do (token/giay):", speeds)

## 4. Xuất file so sánh

File `so_sanh_ban_dich.xlsx` nằm trong thư mục Output của notebook. Các cột `chatgpt`, `gemini` để trống
cho bạn dán vào. Cột `loai_loi` và `ghi_chu` dùng để ghi lỗi khi đọc, ví dụ: sai thuật ngữ, câu gượng,
sai nghĩa, bỏ sót.

In [ ]:
export_xlsx(results).head(10)

## Ghi chú khi đọc kết quả

- Bản dịch ở đây là của mô hình **nén 4-bit**, đúng với thứ người dùng sẽ chạy trên máy cá nhân,
  nên có thể kém hơn một chút so với mô hình gốc không nén.
- Các đoạn công thức, bảng và code đã bị loại khỏi phép thử này. Chất lượng giữ định dạng phải thử riêng
  bằng BabelDOC.
- Nếu bản 27B báo lỗi hết bộ nhớ, giảm `BATCH_SIZE` xuống 2 rồi chạy lại ô dịch. Nếu vẫn lỗi thì
  27B không vừa GPU miễn phí theo cách nén này, và bản 12B sẽ là lựa chọn lớn nhất.
- Nếu bản dịch ra rỗng hoặc toàn ký tự lạ, đổi `torch.bfloat16` thành `torch.float32` trong hàm
  `load_model` rồi chạy lại.